# 01. Baseline 데이터 점검 및 기준 모델

GitHub에 업로드한 사례집 CSV를 원본으로 사용하고, Google Drive에는 데이터 사본과 분석 결과를 저장합니다. DummyClassifier를 통해 최소 기준 성능을 측정합니다.

In [1]:
!pip -q install pandas numpy scikit-learn openpyxl

In [2]:
import sys
from pathlib import Path
from urllib.parse import quote
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/HS-complete-product')
DATA_DIR = PROJECT_DIR / 'data'
RESULTS_DIR = PROJECT_DIR / 'results'
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DATA_FILE = '완제품 모델 데이터_사례_전처리 후 - 사례_전처리 완료.csv'
DATA_URL = 'https://raw.githubusercontent.com/lin1118-code/HS-complete-product/main/' + quote(DATA_FILE)
df = pd.read_csv(DATA_URL, encoding='utf-8-sig')
df.to_csv(DATA_DIR / DATA_FILE, index=False, encoding='utf-8-sig')

print('GitHub 데이터 로드 완료:', df.shape)
print('Drive 데이터 사본 저장:', DATA_DIR / DATA_FILE)
display(df.head())

Mounted at /content/drive
GitHub 데이터 로드 완료: (177, 9)
Drive 데이터 사본 저장: /content/drive/MyDrive/HS-complete-product/data/완제품 모델 데이터_사례_전처리 후 - 사례_전처리 완료.csv


,HS Code,Product Name,Main Ingredients (KR),Main Ingredients (EN),Product Form and Packaging,Intended Use,Product Type,Ruling Summary,Preprocessed Ruling Summary
0,3303.00-1000,Perfumes and scents,NaN,NaN,OO로 조제된 투명한 액상을 롤러 볼 형태의 플라스틱제 용기에 충전하여 종이상자에 ...,"귀밑, 손목 등에 바르는 롤러 타입 향수",향수,"관세율표 제3303호에는 “향수와 화장수”를 분류하며, HSK 제3303.00-10...","관세율표에는 “향수와 화장수”를 분류하며, - 에는 “향수”를 세분류하고 있음 같은..."
1,3303.00-1000,Perfumes and scents,NaN,NaN,OO 등으로 조제된 고상을 펜슬형태의 플라스틱 용기에 넣어 종이상자에 소매포장한 것...,"손목, 팔 안쪽, , 귀 밑에 바르는 고체타입 향수",향수,"관세율표 제3303호에는 “향수와 화장수”를 분류하며, HSK 제3303.00-10...","관세율표에는 “향수와 화장수”를 분류하며, “향수”를 세분류하고 있음 같은 호 HS..."
2,3303.00-2000,TRUE ORIGINAL COLONGE,"변성알코올, 정제수, 향료, 글리세린","Alcohol Denat(Alcohol Denat.), Water, Fragranc...",무색 투명한 액상을 유리제 스프레이식 용기에 담아 종이상자에 소매포장한 것(내용량:...,화장수,화장수,"관세율표 제3303호에는 ""향수와 화장수""가 분류되며, 제3303.00-2000호에...","관세율표 에는 ""향수와 화장수""가 분류되며, ""화장수""를 하고 있음. 같은 호 해설..."
3,3303.00-2000,Musti Eau De Soin Parfumėe,"정제수, PEG-40 수소첨가피마자유, 향료 1.5%, 글리세린, 안식향산나트륨, ...","Water, PEG-40 Hydrogenated Castor Oil, Fragran...",흐린 투명액상을 스프레이식 유리제 용기에 넣어 지제박스에 소매 포장한 것(내용량 5...,영·유아용 화장수,영·유아용 화장수,관세율표 제3303호에는 “향수와 화장수”가 분류되며 112 같은 호 HS해설서에서...,같은 호 HS 해설서에서는 향수와 화장수를 주로 인체에 향기를 주도록 고안된 액체·...
4,3303.00-2000,Toilet waters; BACCART ROUGE 540,"비터오렌지껍질오일, 리모넨, 리날룰, 재스민오일/추출물, 변성알코올, 정제수, PE...","Citrus aurantium peel oil, limonene, linalool,...","투명 액상, 스프레이 유리병 및 지제상자 소매포장(70ml)",Hair perfume,향수·화장수(Hair perfume),향과 알코올 등으로 조제한 화장수로서 소매포장된 물품이므로 통칙 제1호에 따라 제3...,향과 알코올 등으로 조제한 화장수로서 소매포장된 물품


In [3]:
INPUT_COLUMNS = [
    'Product Name',
    'Main Ingredients (KR)',
    'Main Ingredients (EN)',
    'Product Form and Packaging',
    'Intended Use',
]
TARGET_COLUMN = 'HS Code'

required = INPUT_COLUMNS + [TARGET_COLUMN]
missing_columns = [c for c in required if c not in df.columns]
if missing_columns:
    raise ValueError(f'필수 컬럼 누락: {missing_columns}')

for col in INPUT_COLUMNS:
    df[col] = df[col].fillna('').astype(str).str.strip()

df[TARGET_COLUMN] = df[TARGET_COLUMN].astype(str).str.strip()
df = df[df[TARGET_COLUMN].ne('')].reset_index(drop=True)

df['model_text'] = (
    '[PRODUCT] ' + df['Product Name'] +
    ' [INGREDIENTS_KR] ' + df['Main Ingredients (KR)'] +
    ' [INGREDIENTS_EN] ' + df['Main Ingredients (EN)'] +
    ' [FORM] ' + df['Product Form and Packaging'] +
    ' [USE] ' + df['Intended Use']
)

df['duplicate_group'] = pd.util.hash_pandas_object(
    df[INPUT_COLUMNS], index=False
).astype(str)

print('전처리 후 데이터 크기:', df.shape)
display(df[INPUT_COLUMNS + [TARGET_COLUMN]].head())

전처리 후 데이터 크기: (177, 11)


,Product Name,Main Ingredients (KR),Main Ingredients (EN),Product Form and Packaging,Intended Use,HS Code
0,Perfumes and scents,,,OO로 조제된 투명한 액상을 롤러 볼 형태의 플라스틱제 용기에 충전하여 종이상자에 ...,"귀밑, 손목 등에 바르는 롤러 타입 향수",3303.00-1000
1,Perfumes and scents,,,OO 등으로 조제된 고상을 펜슬형태의 플라스틱 용기에 넣어 종이상자에 소매포장한 것...,"손목, 팔 안쪽, , 귀 밑에 바르는 고체타입 향수",3303.00-1000
2,TRUE ORIGINAL COLONGE,"변성알코올, 정제수, 향료, 글리세린","Alcohol Denat(Alcohol Denat.), Water, Fragranc...",무색 투명한 액상을 유리제 스프레이식 용기에 담아 종이상자에 소매포장한 것(내용량:...,화장수,3303.00-2000
3,Musti Eau De Soin Parfumėe,"정제수, PEG-40 수소첨가피마자유, 향료 1.5%, 글리세린, 안식향산나트륨, ...","Water, PEG-40 Hydrogenated Castor Oil, Fragran...",흐린 투명액상을 스프레이식 유리제 용기에 넣어 지제박스에 소매 포장한 것(내용량 5...,영·유아용 화장수,3303.00-2000
4,Toilet waters; BACCART ROUGE 540,"비터오렌지껍질오일, 리모넨, 리날룰, 재스민오일/추출물, 변성알코올, 정제수, PE...","Citrus aurantium peel oil, limonene, linalool,...","투명 액상, 스프레이 유리병 및 지제상자 소매포장(70ml)",Hair perfume,3303.00-2000


## 원본 데이터 중복·그룹 구조 점검

동일한 입력 정보가 여러 HS 코드에 연결되는 충돌 사례가 있는지 먼저 확인합니다. 동일 입력이 여러 행에 반복된 경우에는 같은 그룹으로 묶습니다. 이후 Fold 배정이 가능한지 직접 검사합니다.

In [4]:
conflict_table = (
    df.groupby("duplicate_group")[TARGET_COLUMN]
    .nunique()
    .rename("n_hs_codes")
    .reset_index()
)
conflicting_groups = conflict_table[conflict_table["n_hs_codes"] > 1]
print("동일 입력에 복수 HS 코드가 연결된 그룹:", len(conflicting_groups))
display(conflicting_groups.head(20))

동일 입력에 복수 HS 코드가 연결된 그룹: 0


,duplicate_group,n_hs_codes


## 평가 대상 클래스 선정 및 Fold 가능성 검증

주 평가에는 클래스별 서로 다른 입력 그룹이 최소 4개 있는 경우만 포함합니다. 클래스가 각 Fold의 학습·평가 집합에 가능한 한 모두 나타나는지 사전 검사합니다. Fold 생성이 불가능하면 임의로 진행하지 않고 오류를 표시합니다.

In [5]:
from sklearn.model_selection import StratifiedGroupKFold

class_counts = df[TARGET_COLUMN].value_counts()
eligible_labels = class_counts[class_counts >= 4].index
eval_df = df[df[TARGET_COLUMN].isin(eligible_labels)].copy()

group_counts = eval_df.groupby(TARGET_COLUMN)["duplicate_group"].nunique()
eligible_labels = group_counts[group_counts >= 4].index
eval_df = eval_df[eval_df[TARGET_COLUMN].isin(eligible_labels)].reset_index(drop=True)

print("전체 데이터:", len(df), "건 /", df[TARGET_COLUMN].nunique(), "개 코드")
print("주 평가 데이터:", len(eval_df), "건 /", eval_df[TARGET_COLUMN].nunique(), "개 코드")
print("별도 분석 데이터:", len(df) - len(eval_df), "건")
display(eval_df[TARGET_COLUMN].value_counts().rename("count").to_frame())

if eval_df.empty:
    raise ValueError("평가 가능한 클래스가 없습니다.")

X = eval_df["model_text"]
y = eval_df[TARGET_COLUMN]
groups = eval_df["duplicate_group"]
cv = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=42)
splits = list(cv.split(X, y, groups))

fold_checks = []
for fold, (train_idx, test_idx) in enumerate(splits, start=1):
    train_labels = set(y.iloc[train_idx])
    test_labels = set(y.iloc[test_idx])
    fold_checks.append({
        "fold": fold,
        "train_n": len(train_idx),
        "test_n": len(test_idx),
        "train_classes": len(train_labels),
        "test_classes": len(test_labels),
        "missing_train_classes": sorted(set(y) - train_labels),
        "missing_test_classes": sorted(set(y) - test_labels),
        "group_overlap": len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))
    })

fold_check_df = pd.DataFrame(fold_checks)
display(fold_check_df)
if any(row["group_overlap"] != 0 for row in fold_checks):
    raise ValueError("동일 입력 그룹이 학습·평가 Fold에 중복 배치되었습니다.")

전체 데이터: 177 건 / 31 개 코드
주 평가 데이터: 144 건 / 13 개 코드
별도 분석 데이터: 33 건


,count
HS Code,
3304.99-1000,33
3304.99-9000,33
3307.90-9000,19
3305.90-9000,13
3304.20-9000,8
3307.49-0000,6
3307.30-2000,5
3304.91-1000,5
3306.90-2000,5


,fold,train_n,test_n,train_classes,test_classes,missing_train_classes,missing_test_classes,group_overlap
0,1,110,34,13,12,[],[3305.90-9000],0
1,2,108,36,13,8,[],"[3304.10-9000, 3304.91-1000, 3304.99-3000, 330...",0
2,3,106,38,13,10,[],"[3304.20-9000, 3304.91-1000, 3307.90-4000]",0
3,4,108,36,13,11,[],"[3307.30-2000, 3307.90-4000]",0


## Dummy baseline: 최빈 클래스 예측

DummyClassifier는 제품 정보를 학습하지 않고 학습 Fold에서 가장 많이 등장한 HS 코드만 예측합니다.

In [6]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

fold_results = []
for fold, (train_idx, test_idx) in enumerate(splits, start=1):
    model = DummyClassifier(strategy="most_frequent")
    model.fit(X.iloc[train_idx].to_frame(), y.iloc[train_idx])
    pred = model.predict(X.iloc[test_idx].to_frame())
    fold_results.append({
        "fold": fold,
        "train_n": len(train_idx),
        "test_n": len(test_idx),
        "accuracy": accuracy_score(y.iloc[test_idx], pred),
        "macro_f1": f1_score(y.iloc[test_idx], pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y.iloc[test_idx], pred, average="weighted", zero_division=0),
    })

baseline_results = pd.DataFrame(fold_results)
display(baseline_results)
display(baseline_results[["accuracy", "macro_f1", "weighted_f1"]].agg(["mean", "std"]))

,fold,train_n,test_n,accuracy,macro_f1,weighted_f1
0,1,110,34,0.235294,0.031746,0.089636
1,2,108,36,0.250000,0.050000,0.100000
2,3,106,38,0.210526,0.034783,0.073227
3,4,108,36,0.083333,0.013986,0.012821


,accuracy,macro_f1,weighted_f1
mean,0.194788,0.032629,0.068921
std,0.076068,0.014773,0.038991


In [13]:
from googleapiclient.http import MediaFileUpload

FOLDER_ID = '1GAOGfn5zoLCpgQu62at8WC24gVbCmrma'

for file_path in RESULTS_DIR.glob('*.csv'):
    metadata = {'name': file_path.name, 'parents': [FOLDER_ID]}
    media = MediaFileUpload(str(file_path), mimetype='text/csv', resumable=True)
    result = drive_service.files().create(
        body=metadata,
        media_body=media,
        supportsAllDrives=True,
        fields='id,name'
    ).execute()
    print(f"업로드 완료: {result['name']}")

업로드 완료: baseline_dummy_cv_results.csv
업로드 완료: hs_code_data_audit.csv
업로드 완료: baseline_fold_check.csv
